<a href="https://colab.research.google.com/github/yuchaotaigu/On-Line-Policy-Iteration/blob/main/multidimensional-assignment/On_line_PI_MDA_animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Animation for On-Line Policy Iteration for the Multidimensional Assignment Problem

Animation of the algorithm from:
  "On-Line Policy Iteration for Finite-Horizon Deterministic Optimal Control
   with A Fixed Initial State" (Li, Chen, Li, Fan, Bertsekas, 2026)

Copyright (c) 2026 Yuchao Li

The multidimensional assignment problem (Section 4.1):
  - $N+1$ layers $(N_0, ..., N_{N})$, each with $m$ nodes.
  - A grouping is a set of $(N+1)$ nodes, one per layer.  Its cost depends on the entire tuple $(j_0, j_1, ..., j_{N})$.
  - An $(N+1)$-dimensional assignment partitions all m*N nodes into $m$ disjoint groupings.  Its cost is the sum of grouping costs.
  - Goal: find the minimum-cost assignment.

Formulation as an $N$-stage optimal control problem:
  - Control $u_k$ is a permutation mapping layer-$k$ nodes to layer-$(k+1)$ nodes.
  - State $x_k = (u_0, ..., u_{k-1})$ records the arcs chosen so far.
  - Stage costs $g_k$ = 0 for $k < N$;  terminal cost $g_N(x_N)$ is the assignment cost determined by $(u_0, ..., u_{N-1})$.
  - A policy $\pi^\ell$ is represented by the stored trajectory; the generator $G$ replays it (consistent by construction).

Key insight for the minimization at stage k:
  Given the prefix $(u_0^{\ell+1}, ..., u_{k-1}^{\ell+1})$ and the suffix from policy $\pi^\ell$ as $(u_{k+1}^\ell, ..., u_{N-2}^\ell)$, the only free variable is the
  permutation $u_k$.  The total cost decomposes as a sum over $m$ groupings, and each grouping's contribution depends only on which node it visits at layer $k+1$ (the rest of the path is fixed).  So choosing $u_k$ amounts to
  solving a bipartite matching (linear assignment) problem, where the weight of assigning node a at layer $k$ to node $b$ at layer $k+1$ equals the grouping cost of the path that passes through a at layer $k$ and $b$ at layer $k+1$.

In [1]:
import math
import itertools
import io

import numpy as np
from scipy.optimize import linear_sum_assignment
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

# Problem Definition & Algorithm

In [2]:
def random_grouping_costs(N: int, m: int, seed: int = 0) -> np.ndarray:
    rng = np.random.RandomState(seed)
    return rng.uniform(0, 100, size=(m,) * (N + 1))


def assignment_cost(costs: np.ndarray, perms: list[np.ndarray]) -> float:
    m = costs.shape[0]
    num_layers = costs.ndim
    total = 0.0
    for i in range(m):
        idx = [0] * num_layers
        idx[0] = i
        node = i
        for k in range(num_layers - 1):
            node = perms[k][node]
            idx[k + 1] = node
        total += costs[tuple(idx)]
    return total


def get_trajectories(perms, m):
    trajs = []
    for i in range(m):
        path = [i]
        node = i
        for perm in perms:
            node = perm[node]
            path.append(node)
        trajs.append(path)
    return trajs


def online_pi_iteration_staged(costs, perms):
    """
    One PI iteration returning intermediate states after each stage update.
    Returns list of (full_perms, stage_index) and the final new_perms.
    """
    N = len(perms)
    m = costs.shape[0]
    new_perms = []
    stages = []

    for k in range(N):
        node_at_k = np.zeros(m, dtype=int)
        for i in range(m):
            node = i
            for s in range(k):
                node = new_perms[s][node]
            node_at_k[i] = node

        suffix_paths = {}
        for b in range(m):
            path = [b]
            node = b
            for s in range(k + 1, N):
                node = perms[s][node]
                path.append(node)
            suffix_paths[b] = path

        prefix_paths = []
        for i in range(m):
            prefix = [i]
            node = i
            for s in range(k):
                node = new_perms[s][node]
                prefix.append(node)
            prefix_paths.append(prefix)

        C_grouping = np.zeros((m, m))
        for i in range(m):
            for b in range(m):
                full_path = prefix_paths[i] + suffix_paths[b]
                C_grouping[i, b] = costs[tuple(full_path)]

        inv_node_at_k = np.zeros(m, dtype=int)
        for i in range(m):
            inv_node_at_k[node_at_k[i]] = i

        C_matching = np.zeros((m, m))
        for a in range(m):
            C_matching[a, :] = C_grouping[inv_node_at_k[a], :]

        row_ind, col_ind = linear_sum_assignment(C_matching)
        new_perm = np.zeros(m, dtype=int)
        for a, b in zip(row_ind, col_ind):
            new_perm[a] = b
        new_perms.append(new_perm)

        full = [p.copy() for p in new_perms] + [p.copy() for p in perms[k + 1:]]
        stages.append((full, k))

    return stages, new_perms


def brute_force_optimal(costs):
    N = costs.ndim - 1
    m = costs.shape[0]
    all_perms = [np.array(p, dtype=int) for p in itertools.permutations(range(m))]
    best_cost = np.inf
    for combo in itertools.product(all_perms, repeat=N):
        c = assignment_cost(costs, list(combo))
        if c < best_cost:
            best_cost = c
    return best_cost

# Drawing Nodes & Animation

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Drawing
# ══════════════════════════════════════════════════════════════════════════════

COLORS = [
    "#C4462B",  # warm red
    "#2D5F8A",  # steel blue
    "#3A8A5C",  # forest green
    "#C68B28",  # goldenrod
    "#7B5EA7",  # purple
    "#D4737E",  # rose
    "#3C8A8A",  # teal
    "#B85C2A",  # rust
]


def fig_to_pil(fig, dpi=120):
    """Render a matplotlib figure to a PIL Image."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, facecolor=fig.get_facecolor())
    buf.seek(0)
    img = Image.open(buf).convert("RGB")
    return img


def draw_frame(fig, ax, N, m, trajs, updated_stages, title_text, cost_text,
               opt_text=None):
    """
    Draw one animation frame.

    Parameters
    ----------
    trajs : list of m trajectories, each a list of N+1 node indices
    updated_stages : int in 0..N — how many stages have been updated.
        Edges for layers 0..updated_stages-1 are solid colored arrows;
        edges for layers updated_stages..N-1 are dashed gray.
    """
    ax.clear()

    # Layout
    x_margin = 0.6
    ax.set_xlim(-x_margin, N + x_margin)
    y_lo, y_hi = -0.5, m - 0.5
    y_span = y_hi - y_lo
    ax.set_ylim(y_hi + 0.32 * y_span, y_lo - 0.15 * y_span)
    ax.set_aspect("equal")
    ax.axis("off")

    # ── Title (top center) ──────────────────────────────────────────────
    ax.text(
        N / 2, y_lo - 0.13 * y_span, title_text,
        ha="center", va="bottom", fontsize=12.5, fontweight="bold",
        color="#1A1A1A",
    )
    if opt_text:
        ax.text(
            N / 2, y_lo - 0.06 * y_span, opt_text,
            ha="center", va="bottom", fontsize=9.5, color="#555555",
        )

    # ── Cost (bottom center, blue) ─────────────────────────────────────
    ax.text(
        N / 2, y_hi + 0.24 * y_span, cost_text,
        ha="center", va="bottom", fontsize=11, fontweight="bold",
        color="#2255AA",
    )

    # ── Layer labels ───────────────────────────────────────────────────
    for k in range(N + 1):
        ax.text(
            k, y_hi + 0.15 * y_span, f"Layer {k}",
            ha="center", va="bottom", fontsize=8, color="#777777",
        )

    # ── Edges ──────────────────────────────────────────────────────────
    for i, traj in enumerate(trajs):
        color = COLORS[i % len(COLORS)]
        for k in range(N):
            x0, y0 = k, traj[k]
            x1, y1 = k + 1, traj[k + 1]

            if k < updated_stages:
                # Updated: solid colored arrow
                ax.annotate(
                    "", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=color,
                        lw=2.0,
                        shrinkA=6, shrinkB=6,
                        mutation_scale=10,
                    ),
                    zorder=2,
                )
            else:
                # Not yet updated: dashed gray
                ax.plot(
                    [x0, x1], [y0, y1],
                    color="#AAAAAA", linewidth=1.0,
                    linestyle="--", dash_capstyle="round",
                    zorder=1,
                )

    # ── Nodes ──────────────────────────────────────────────────────────
    for k in range(N + 1):
        for j in range(m):
            in_updated = (k <= updated_stages) and (updated_stages > 0)
            ec = "#333333" if in_updated else "#999999"
            lw = 1.4 if in_updated else 1.0
            circle = plt.Circle(
                (k, j), 0.14,
                facecolor="white", edgecolor=ec, linewidth=lw, zorder=5,
            )
            ax.add_patch(circle)
            ax.text(
                k, j, str(j),
                ha="center", va="center", fontsize=7.5,
                color="#333333", fontweight="medium", zorder=6,
            )

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Animation generation
# ══════════════════════════════════════════════════════════════════════════════

def generate_animation(
    N=5,
    m=4,
    seed=0,
    init_seed=None,
    max_iters=10,
    output="online_pi_animation.gif",
    dpi=120,
    ms_stage=800,
    ms_iter_end=1800,
    ms_initial=2500,
):
    """
    Generate a GIF animation of on-line PI for MDA.

    Parameters
    ----------
    N, m, seed, init_seed : problem parameters
    max_iters : maximum PI iterations
    output : output GIF path
    dpi : image resolution
    ms_stage : frame duration (ms) for each stage-update frame
    ms_iter_end : frame duration (ms) after all stages in one iteration
    ms_initial : frame duration (ms) for the initial and converged frames
    """
    if init_seed is None:
        init_seed = seed + 10000

    costs = random_grouping_costs(N, m, seed=seed)

    # Optimal cost for small instances
    opt_cost = None
    if math.factorial(m) ** N <= 500_000:
        print("Computing optimal cost via brute force...")
        opt_cost = brute_force_optimal(costs)
        print(f"  J* = {opt_cost:.2f}")

    opt_text = f"Optimal Cost J* = {opt_cost:.2f}" if opt_cost is not None else None

    # Initial random policy
    rng = np.random.RandomState(init_seed)
    perms = [rng.permutation(m).astype(int) for _ in range(N)]

    # ── Collect frame specifications ────────────────────────────────────
    #    Each entry: (trajs, updated_stages, title, cost_text, duration_ms)
    frame_specs = []

    trajs = get_trajectories(perms, m)
    init_cost = assignment_cost(costs, perms)
    frame_specs.append((
        trajs, 0,
        "Initial Policy (random)",
        f"Total Cost: {init_cost:.2f}",
        ms_initial,
    ))

    for it in range(1, max_iters + 1):
        stages, new_perms = online_pi_iteration_staged(costs, perms)

        for idx_s, (full_perms, k) in enumerate(stages):
            trajs_k = get_trajectories(full_perms, m)
            cost_k = assignment_cost(costs, full_perms)
            is_last_stage = (idx_s == len(stages) - 1)
            dur = ms_iter_end if is_last_stage else ms_stage

            title = (
                f"Iteration {it}:  "
                f"update $u_{k}$  (layer {k} \u2192 {k+1})"
            )

            frame_specs.append((
                trajs_k, k + 1,
                title,
                f"Total Cost: {cost_k:.2f}",
                dur,
            ))

        new_cost = assignment_cost(costs, new_perms)
        old_cost = assignment_cost(costs, perms)
        perms = [p.copy() for p in new_perms]

        if abs(new_cost - old_cost) < 1e-12:
            trajs_final = get_trajectories(perms, m)
            frame_specs.append((
                trajs_final, N,
                f"Converged at iteration {it}",
                f"Final Cost: {new_cost:.2f}",
                ms_initial,
            ))
            break

    print(f"Total frames: {len(frame_specs)}")

    # ── Render frames ───────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor("white")
    plt.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.04)

    pil_frames = []
    durations = []

    for i, (tr, upd, ttl, ct, dur) in enumerate(frame_specs):
        draw_frame(fig, ax, N, m, tr, upd, ttl, ct, opt_text=opt_text)
        pil_frames.append(fig_to_pil(fig, dpi=dpi))
        durations.append(dur)
        print(f"  [{i+1:>2}/{len(frame_specs)}]  {ttl}")

    plt.close(fig)

    # ── Assemble GIF ────────────────────────────────────────────────────
    pil_frames[0].save(
        output,
        save_all=True,
        append_images=pil_frames[1:],
        duration=durations,
        loop=0,
    )

    file_kb = round(
        sum(f.size[0] * f.size[1] for f in pil_frames) * 3 / 1024 / len(pil_frames)
    )
    print(f"\nSaved: {output}")
    print(f"  {len(pil_frames)} frames, {sum(durations)/1000:.1f}s total duration")

# GIF Generation

In [6]:
if __name__ == "__main__":
    generate_animation(
        N=5, m=4, seed=0, init_seed=10000,
        max_iters=10,
        output="Online_PI_MDA_animation.gif",
    )

Total frames: 27
  [ 1/27]  Initial Policy (random)
  [ 2/27]  Iteration 1:  update $u_0$  (layer 0 → 1)
  [ 3/27]  Iteration 1:  update $u_1$  (layer 1 → 2)
  [ 4/27]  Iteration 1:  update $u_2$  (layer 2 → 3)
  [ 5/27]  Iteration 1:  update $u_3$  (layer 3 → 4)
  [ 6/27]  Iteration 1:  update $u_4$  (layer 4 → 5)
  [ 7/27]  Iteration 2:  update $u_0$  (layer 0 → 1)
  [ 8/27]  Iteration 2:  update $u_1$  (layer 1 → 2)
  [ 9/27]  Iteration 2:  update $u_2$  (layer 2 → 3)
  [10/27]  Iteration 2:  update $u_3$  (layer 3 → 4)
  [11/27]  Iteration 2:  update $u_4$  (layer 4 → 5)
  [12/27]  Iteration 3:  update $u_0$  (layer 0 → 1)
  [13/27]  Iteration 3:  update $u_1$  (layer 1 → 2)
  [14/27]  Iteration 3:  update $u_2$  (layer 2 → 3)
  [15/27]  Iteration 3:  update $u_3$  (layer 3 → 4)
  [16/27]  Iteration 3:  update $u_4$  (layer 4 → 5)
  [17/27]  Iteration 4:  update $u_0$  (layer 0 → 1)
  [18/27]  Iteration 4:  update $u_1$  (layer 1 → 2)
  [19/27]  Iteration 4:  update $u_2$  (layer 2